In [ ]:
# claude --resume "rename-generative-backend-to-llmr"

In [1]:
import subprocess
result = subprocess.run(["python", "-c", "from geometry_msgs.msg import PointStamped; print('OK')"], capture_output=True, text=True)
print(result.stdout, result.stderr)

OK
 


In [2]:
import logging
import pathlib
import threading

from dotenv import load_dotenv

from pycram.language import SequentialPlan
# ── pycram ─────────────────────────────────────────────────────────────────────
from pycram.motion_executor import simulated_robot

# ── Generative backend ─────────────────────────────────────────────────────────
from llmr import ExecutionLoop, ActionPipeline, TaskDecomposer
from llmr import load_pr2_apartment_world
from llmr.pipeline import ActionDispatcher

# Load API keys before any LLM call
_env = pathlib.Path().resolve().parent / ".env"
load_dotenv(_env, override=True)
import logging

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("llmr").setLevel(logging.DEBUG)

print("All imports OK")

Found these qp solvers: ['qpSWIFT', 'qpalm']


All imports OK


In [3]:
# Build world (parses URDFs fresh — semantic annotations preserved)
world, pr2, context = load_pr2_apartment_world()
print("World loaded")

Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/base_footprint
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/base_footprint
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/base_link
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/base_link
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr

World loaded


In [4]:
world.__dict__.keys()

dict_keys(['simulator_additional_properties', 'kinematic_structure', 'semantic_annotations', 'degrees_of_freedom', 'actuators', 'world_is_being_modified', 'name', '_id', '_model_manager', '_world_entity_hash_table', 'state', '_forward_kinematic_manager', 'collision_manager', '_atomic_modification_is_being_executed'])

In [5]:
len(world.semantic_annotations)

15

In [6]:
import rclpy
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
from llmr import ExecutionLoop, RecoveryHandler

rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
import threading
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

In [7]:
_tf_publisher = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print("ROS2 publishers started")

ROS2 publishers started


In [8]:
def reset_world():
    """Reset world to base state without restarting the kernel."""
    global world, pr2, context, _tf_publisher, _viz_publisher
    world, pr2, context = load_pr2_apartment_world()
    _tf_publisher = TFPublisher(_world=world, node=_ros_node)
    _viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
    print("World reset OK")


print("reset_world() ready — call it to restore the world between runs")

reset_world() ready — call it to restore the world between runs


In [9]:
pipeline = ActionPipeline.from_world(world)
print("ActionPipeline ready")
print(f"  manipulator      : {type(pipeline.world_context.manipulator).__name__}")
print(f"  registered types : {list(ActionDispatcher._registry.keys())}")

ActionPipeline ready
  manipulator      : ParallelGripper
  registered types : ['PickUpAction', 'PlaceAction']


In [10]:
pipeline.__dict__['world_context'].__dict__.keys()

dict_keys(['manipulator'])

In [11]:
handler = RecoveryHandler(world=world, max_retries=2)

loop = ExecutionLoop(
    world=world,
    pipeline=pipeline,
    context=context,
    robot_context=lambda: simulated_robot,
    decomposer=TaskDecomposer(),
    recovery_handler=handler
)
print("ExecutionLoop ready")

ExecutionLoop ready


In [12]:
# # Reset world to a clean state before running the loop
# reset_world()
# pipeline = ActionPipeline.from_world(world)
# loop = ExecutionLoop(
#     world=world,
#     pipeline=pipeline,
#     context=context,
#     robot_context=lambda: simulated_robot,
#     decomposer=TaskDecomposer(),
# )
# print("World reset and loop ready")

In [13]:
results = loop.run([
# "fetch the milk and place it on the island_countertop",
#     "pick up the breakfast_cereal from the counter",
        "Pick up the milk from the counter",
# "Pick up the mug from the counter",
# "Place the milk on the island_countertop",
# "pick up the breakfast_cereal from the counter",
#     "Place the breakfast_cereal on the island_countertop",
# "Place the milk on the island_countertop",
#     "Pick up the milk from the counter",
])

INFO:task_decomposer.py::299 decompose Decomposed 'Pick up the milk from the counter' → steps=['pick up the milk from the counter'] deps={}
DEBUG:execution_loop.py::169 run ExecutionLoop: 'Pick up the milk from the counter' → 1 sub-instructions, deps={}
INFO:execution_loop.py::201 run ExecutionLoop: planning 'pick up the milk from the counter'
INFO:action_pipeline.py::261 run ActionPipeline.run: 'pick up the milk from the counter'
DEBUG:action_pipeline.py::281 classify_and_extract World context for slot filling:
## World State Summary

Scene objects and surfaces: base_footprint, map, odom_combined, apartment_root, furnitures_root, walls, windows, wall_coloksu_wall1, wall_coloksu_wall2, wall_coloksu_wall3, wall_coloksu_wall4, wardrobe, wardrobe_door_left, wardrobe_door_left_handle, wardrobe_door_right, wardrobe_door_right_handle, armchair, sofa, coffee_table, coffee_table_drawer, bedside_table, kitchen_root, side_A, side_B, cabinet1, cabinet1_door_top_left, handle_cab1_top_door, oven, c

In [ ]:
for r in results:
    status = "OK" if r.success else f"FAILED: {r.error}"
    actions = [type(a).__name__ for a in [*r.preconditions, r.action]] if r.action else []
    print(f"{r.instruction!r}  →  {status}")
    if actions:
        print(f"  actions: {actions}")

In [ ]:

results[0].__dict__.keys()

In [ ]:
results[-1]

In [ ]:
world.get_body_by_name('island_countertop').global_pose.to_json()

In [ ]:
# result = loop.run_single("Pick up the mug")
# if result.clarification:
#   req = result.clarification
#   print(f"Need clarification for {req.entity_role} '{req.entity_name}'")
#   print(f"Did you mean one of: {req.available_names[-10:]}")

## Testing

In [ ]:
from pycram.datastructures.enums import Arms, ApproachDirection, VerticalAlignment
from pycram.datastructures.grasp import GraspDescription
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.robot_plans import (
    ParkArmsActionDescription,
    MoveTorsoActionDescription,
    NavigateActionDescription,
    PickUpActionDescription,
    PlaceActionDescription,
)
from semantic_digital_twin.datastructures.definitions import TorsoState
from pycram.language import SequentialPlan
from pycram.view_manager import ViewManager
from pycram.datastructures.pose import PoseStamped
from pycram.designators.location_designator import CostmapLocation, SemanticCostmapLocation
#
# cereal_body = world.get_body_by_name("breakfast_cereal.stl")
# cereal_pose  = PoseStamped.from_spatial_type(cereal_body.global_pose)
# arm_view    = ViewManager.get_arm_view(Arms.RIGHT, pr2)
# nav_loc = CostmapLocation(target=cereal_pose, reachable_for=pr2, reachable_arm=Arms.RIGHT)
#
# grasp = GraspDescription(
#     approach_direction=ApproachDirection.FRONT,
#     vertical_alignment=VerticalAlignment.TOP,
#     rotate_gripper=False,
#     manipulator=arm_view.manipulator
# )
#
#
# action = PickUpAction(
#   object_designator=cereal_body,
#   arm=Arms.RIGHT,
#   grasp_description=grasp,
# )
#
# # Execute directly (no navigation preconditions)
# with simulated_robot:
#   SequentialPlan(context,
#                  ParkArmsActionDescription(Arms.BOTH),
#                 MoveTorsoActionDescription([TorsoState.HIGH]),
#                  NavigateActionDescription(nav_loc),
#                  action).perform()

In [ ]:
milk = world.get_body_by_name("milk.stl")
cereal = world.get_body_by_name("breakfast_cereal.stl")
milk_nav_pose = CostmapLocation(target=milk, reachable_for=pr2, reachable_arm=Arms.RIGHT)
cereal_nav_pose = CostmapLocation(target=cereal, reachable_for=pr2, reachable_arm=Arms.RIGHT)

with simulated_robot:
    SequentialPlan(context,
                   MoveTorsoActionDescription([TorsoState.LOW]),
                   MoveTorsoActionDescription([TorsoState.HIGH]),
                   NavigateActionDescription(cereal_nav_pose)).perform()

In [ ]:
table = world.get_body_by_name("island_countertop")
table_sem_loc = SemanticCostmapLocation(body=table, for_object=cereal)
table_sem_loc_ground = table_sem_loc.ground()
table_nav_pose = CostmapLocation(target=PoseStamped.from_list(position=[table_sem_loc_ground.pose.position.x,
                                                                        table_sem_loc_ground.pose.position.y,
                                                                        0],
                                                              orientation=[table_sem_loc_ground.pose.orientation.x,
                                                                           table_sem_loc_ground.pose.orientation.y,
                                                                           table_sem_loc_ground.pose.orientation.z,
                                                                           table_sem_loc_ground.pose.orientation.w], frame=world.root), reachable_for=pr2, reachable_arm=Arms.RIGHT)

In [ ]:
with simulated_robot:
    SequentialPlan(context,NavigateActionDescription(table_nav_pose)).perform()

In [ ]:
table_sem_loc_ground.pose.position.x